# Biodiversity 

In [ ]:
from matplotlib import pyplot as plt
import contextily as ctx
import geopandas as gpd
import geoviews as gv
import hvplot.pandas
import pandas as pd
import fiona
import os

### load data

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')

gdb_path = os.path.join(data_dir, 'Lebensraumkarte_v1_1_VD_20241025.gdb')
layers = fiona.listlayers(gdb_path)
areas = gpd.read_file(gdb_path, layer=layers[0], columns=['TypoCH_NUM', 'TypoCH', 'POLYID', 'geometry'])

roads_path = os.path.join(data_dir, 'lausanne_drive_edges.gpkg')
roads = gpd.read_file(roads_path, columns=['geometry'])

### Select biodiversity areas within Lausanne

In [ ]:
buffer_distance = 10  # meters

# select only biodiversity areas within Lausanne
lausanne_boundary = gpd.read_file(os.path.join(data_dir, 'Lausanne Districts.gpkg'), columns=['geometry'])
lausanne_boundary["geometry"] = lausanne_boundary.geometry.buffer(buffer_distance)  # includes outside areas that are close to the boundary
xmin, ymin, xmax, ymax = lausanne_boundary.total_bounds

# Clip the areas to the bounding box of Lausanne and remove unwanted areas
areas_lausanne = areas.cx[xmin:xmax, ymin:ymax]
areas_lausanne = areas_lausanne[~areas_lausanne["TypoCH"].str.startswith("9")] # Zones spéciales
areas_lausanne = areas_lausanne[~areas_lausanne["TypoCH"].str.startswith("8")] # Zones de trasport
areas_lausanne = areas_lausanne[~areas_lausanne["TypoCH"].str.startswith("5")] # Surfaces bâties
areas_lausanne = areas_lausanne[areas_lausanne["TypoCH"] != "4"] # Surfaces ouvertes
areas_lausanne = areas_lausanne[areas_lausanne["POLYID"] != 21691543] # Lac
areas_lausanne = areas_lausanne[~areas_lausanne["TypoCH"].str.startswith("STR")] # STR

### Find biodiversity areas within a certain buffer distance of roads

In [ ]:
# Ensure same CRS by reprojecting roads to match areas_lausanne
roads = roads.to_crs(areas_lausanne.crs)

# Create a buffered version of roads without modifying the original
roads_buffered = roads.copy()
roads_buffered["road_id"] = roads_buffered.index
roads_buffered['geometry'] = roads_buffered.geometry.buffer(buffer_distance)

# Spatial join - this will give you road geometry
roads_areas = gpd.sjoin(roads_buffered, areas_lausanne, how='inner', predicate='intersects')

# To keep BOTH geometries, rename the road geometry and add area geometry
roads_areas = roads_areas.rename(columns={'geometry': 'road_geometry'})
# Add the area geometry by merging with original areas
roads_areas = roads_areas.merge(
    areas_lausanne[['POLYID', 'geometry']], 
    on='POLYID', 
    how='left'
).rename(columns={'geometry': 'area_geometry'})

roads_areas.drop(columns=['index_right'], inplace=True)

# Set which geometry you want as the active geometry (area in your case)
roads_areas = roads_areas.set_geometry('area_geometry')

road_centroids_x = roads_areas['road_geometry'].centroid.x
area_centroids_x = roads_areas['area_geometry'].centroid.x
roads_areas["side"] = (road_centroids_x < area_centroids_x).map({True: 'left', False: 'right'})

#roads_areas["cluster_v1"] = pd.NA # Clusters based on the exact TypoCH code
roads_areas["cluster_v2"] = pd.NA # Clusters based on the first 3 characters of the TypoCH code (more general)

#roads_areas["cluster_v1"] = roads_areas["cluster_v1"].astype("Int64")
roads_areas["cluster_v2"] = roads_areas["cluster_v2"].astype("Int64")

roads_areas["TypoCH_v2"] = roads_areas["TypoCH"].str[:3]

### Create the clusters

In [ ]:
#cluster = 0
#
#for (road_id, typo_ch), group_df in roads_areas.groupby(['road_id', 'TypoCH']):
#    if 'right' in group_df['side'].values and 'left' in group_df['side'].values:
#        roads_areas.loc[group_df.index, 'cluster_v1'] = cluster
#        cluster += 1

cluster = 0

for (road_id, typo_ch), group_df in roads_areas.groupby(['road_id', 'TypoCH_v2']):
    if 'right' in group_df['side'].values and 'left' in group_df['side'].values:
        roads_areas.loc[group_df.index, 'cluster_v2'] = cluster
        cluster += 1

In [ ]:
#roads_areas.columns
roads_areas["TypoCH_v2"].unique()

### Regroup the clusters

In [ ]:
done = False

while not done:

    done = True
    for polyid, group_df in roads_areas.groupby('POLYID'):
        if group_df['cluster_v2'].nunique(dropna = True) > 1:

            done = False

            final_cluster = group_df['cluster_v2'].iloc[0]
            
            if pd.isna(final_cluster):
                final_cluster = group_df['cluster_v2'].iloc[1]
            
            for cluster_id in group_df['cluster_v2'].unique():
                if pd.isna(cluster_id):
                    continue
                roads_areas.loc[roads_areas['cluster_v2'] == cluster_id, 'cluster_v2'] = final_cluster
            break

### Clean data

In [ ]:
# Drop duplicate rows based on road_id and POLYID combination
roads_areas = roads_areas.drop_duplicates(subset=['road_id', 'POLYID'], keep='first')
roads_areas = roads_areas.drop_duplicates(subset=['cluster_v2', 'POLYID'], keep='first')

roads_areas = roads_areas[roads_areas["cluster_v2"].notna()]

### Save data

In [ ]:
roads_areas["road_geometry"] = roads_areas["road_geometry"].to_wkt()
roads_areas.to_file('roads_areas_clusters.gpkg', layer='roads_areas_clusters', driver='GPKG')

### Plot data

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))

# Plot roads_areas with cluster_v2 column
roads_areas.to_crs(epsg=3857).plot(
    ax=ax,
    column='cluster_v2',
    categorical=True,
    legend=False,
    alpha=0.7,
    cmap='tab20',
    zorder=7,
    legend_kwds={"loc": "upper left", "bbox_to_anchor": (1, 1)}
)

# Add OpenStreetMap basemap
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.CH, zoom=14)

plt.title("Biodiversity Areas Clusters (v2)")
plt.tight_layout()
plt.show()

### Prepare stats for tooltip

In [ ]:
roads_areas['area'] = roads_areas.geometry.area

roads_areas['total_area'] = roads_areas.groupby('cluster_v2')['area'].transform('sum')
roads_areas['max_area'] = roads_areas.groupby('cluster_v2')['area'].transform('max')

roads_areas['gain_marginal'] = roads_areas['total_area'] / roads_areas['max_area']

roads_areas["road_amount"] = roads_areas.groupby('cluster_v2')["road_id"].transform('nunique')

In [ ]:
# Interactive plot with basemap
cluster_plots = roads_areas[roads_areas['cluster_v2'].notna()].hvplot(
    c='cluster_v2',
    cmap='tab20',
    alpha=0.7,
    tiles='OSM',
    frame_width=800,
    frame_height=800,
    legend=False,
    title="Biodiversity Areas Clusters (v2)",
    hover_cols=['POLYID', 'TypoCH_v2', 'total_area', 'max_area', 'gain_marginal', 'road_amount']
) 

roads_areas["road_geometry"] = gpd.GeoSeries.from_wkt(roads_areas["road_geometry"], crs=roads.crs).to_crs(roads_areas.crs)

roads_plot = roads_areas.set_geometry('road_geometry').hvplot(
#roads_plot= roads.hvplot(
    line_color='black',
    line_width=1,
    alpha=0.7,
    hover=False
)

cluster_plots * roads_plot